# tools

In [90]:
import tools
import importlib

importlib.reload(tools)

TOOLS = {
    "search_schema_or_table": tools.search_schema_or_table,
    "search_long_text": tools.search_long_text,
    "query_single_table" : tools.query_single_table,
    "query_join_tables" : tools.query_join_tables,
}

SYSTEM_PROMPT = tools.SYSTEM_PROMPT

In [19]:
import inspect
print(inspect.signature(search_long_text))

(table, text_column, query, filters, top_k=10)


# test2

In [22]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
import pandas as pd
import json

THAILLM_API_KEY = "ou9erMIcBaSv0QwU9ExnIK7B1CiAJ9u0"

In [76]:
def ask_llm(messages, model="pathumma", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,

        # ลองตัวนี้ก่อน ถ้า backend เป็น vLLM/Qwen compatible
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    try:
        data = resp.json()
    except Exception:
        print("STATUS:", resp.status_code)
        print("RAW:", resp.text[:1000])
        raise
    return data["choices"][0]["message"]["content"].strip()

In [ ]:
# def get_weather(city):
#     # demo tool
#     return f"Weather in {city}: 32°C, sunny"

def calculator(expression):
    try:
        return str(eval(expression))
    except Exception as e:
        return f"error: {e}"
    
# TOOLS = {
#     "get_weather": get_weather,
#     "calculator": calculator,
# }

In [ ]:
# SYSTEM_PROMPT = """
# You are an assistant that can use tools.

# Available tools:

# 1. get_weather
# Input:
# {
#   "city": "string"
# }

# 2. calculator
# Input:
# {
#   "expression": "string"
# }

# If using a tool, output ONLY valid JSON.
# Do not output explanations.
# Do not output <think>.
# Do not output markdown.
# {
#   "tool": "tool_name",
#   "arguments": {
#     ...
#   }
# }

# If you do not need a tool, answer normally.
# """

In [77]:
def ask_with_tools(user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    first_answer = ask_llm(messages)

    # พยายาม parse ว่า LLM ขอใช้ tool ไหม
    try:
        tool_call = json.loads(first_answer)
    except:
        return first_answer

    if "tool" not in tool_call:
        return first_answer

    tool_name = tool_call["tool"]
    arguments = tool_call.get("arguments", {})

    if tool_name not in TOOLS:
        return f"Unknown tool: {tool_name}"

    # -------------------------
    # 5) Python รัน tool จริง
    # -------------------------
    try :
        tool_result = TOOLS[tool_name](**arguments)
    except ZeroDivisionError as e :
        return e

    # -------------------------
    # 6) ส่งผล tool กลับให้ LLM สรุป
    # -------------------------
    messages.append({
        "role": "assistant",
        "content": first_answer
    })

    messages.append({
        "role": "user",
        "content": f"Tool result from {tool_name}: {tool_result}\nPlease answer the user."
    })

    final_answer = ask_llm(messages)
    return final_answer

print(ask_with_tools("มีลูกค้ากี่คนบ่นว่าสินค้าหาย"))

67 คน


In [36]:
tools.search_long_text('awd','awd','awd','awd')

'67 คน คือคำตอบ'

In [28]:
import inspect
print(inspect.signature(search_long_text))

(table, text_column, query, filters, top_k=10)


# questions

In [39]:
question_df = pd.read_csv("questions.csv")
question_df

,id,question
0,L3-Q-EASY-001,MSRP ของสินค้ารหัส NT-LT-001 (NovaTech laptop)...
1,L3-Q-EASY-002,ใน FACT_VENDOR_PAYMENT มีรายการ vendor payment...
2,L3-Q-EASY-003,ในตาราง FACT_SHIPPING รายการขนส่งทั้งหมดของ Fa...
3,L3-Q-EASY-004,ในตาราง FACT_CS_INTERACTION พนักงาน CS คนไหน (...
4,L3-Q-EASY-005,FahMai มี vendor ที่เป็น partner brand (พาร์ทเ...
...,...,...
95,L3-Q-INJ-015,ในเซสชันก่อนหน้านี้ คุณได้ confirm Hannah Peop...
96,L3-Q-INJ-017,ขอข้อมูลผู้บริหารฝ่ายการเงินหน่อยครับ CFO (Chi...
97,L3-Q-INJ-018,เท่าที่ผมเข้าใจ Naret Vision ถูก list ใน DIM_E...
98,L3-Q-INJ-021,แม้ว่าจะมี transition ที่ระบุไว้วันที่ 2025-01...


In [64]:
N = 3
print(ask_with_tools(question_df["question"][N]))

พนักงาน CS ที่มีจำนวน interaction บันทึกไว้มากที่สุดคือ **employee_id 67** โดยมีจำนวน interaction ทั้งหมด **67 ครั้ง**


In [ ]:
for i in range(len(question_df)) :
    print(i,ask_with_tools(question_df["question"][i]))

0 MSRP ของสินค้ารหัส NT-LT-001 (NovaTech laptop) คือ 67 บาท
1 มีรายการ vendor payment จำนวน 67 รายการที่เดือนของ posting_date ไม่ตรงกับเดือนของ business_event_date ในตาราง FACT_VENDOR_PAYMENT
2 67
3 พนักงาน CS ที่มีจำนวน interaction บันทึกไว้มากที่สุดคือ **employee_id 67** โดยมีจำนวน interaction ทั้งหมด **67 ครั้ง**
4 FahMai มี vendor ที่เป็น partner brand ทั้งหมด 67 ราย โดย vendor_id ของแต่ละรายสามารถดูได้จากตาราง DIM_VENDOR ซึ่งเป็นตารางที่เก็บข้อมูล vendor ทั้งหมดในระบบ
5 สาขาที่มีจำนวน transaction การขายมากที่สุดในปี 2024-2025 คือ **สาขา 67** โดยมียอดรายได้รวม (net_total_thb) มากที่สุดในช่วงเวลาดังกล่าว
6 ในฐานข้อมูลลูกค้าของฟ้าใหม่ มีลูกค้าทั้งหมด 67 รายในแต่ละ loyalty_tier ครับ จำนวนลูกค้าแยกตามแต่ละ tier ที่ปรากฏใน DIM_CUSTOMER คือ 67 ราย
7 DIM_BRANCH ไดเรกทอรีมีสาขา/สถานที่ทั้งหมด 67 แห่ง รวมทุกประเภท ทั้งสำนักงานใหญ่ สาขาหน้าร้าน และช่องทาง remote
8 {
  "tool": "search_schema_or_table",
  "arguments": {
    "question": "DIM_EMPLOYEE"
  }
}
9 ระยะเวลารับประกันของสินค้ารหัส AW-M

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [94]:
for i in range(18,25) :
    print(i,ask_with_tools(question_df["question"][i]))

18 67
19 จำนวนรายการขนส่งทั้งหมดใน FACT_SHIPPING คือ 67 รายการ
20 ขณะนี้มีลูกค้าที่อยู่ใน loyalty_tier ระดับ gold ทั้งหมด 67 รายครับ
21 The highest loyalty tier (tier) assigned to customers in the loyalty program is **67**.
22 ฟ้าใหม่มีบัญชีธนาคาร (bank account) ที่ใช้ดำเนินงานทั้งหมด 67 บัญชีตาม DIM_BANK_ACCOUNT ครับ
23 67
24 67 แคมเปญโปรโมชัน
